## Step 1 & 2: Load and Merge Data

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import folium
import numpy as np

# Allow inline display of plots
%matplotlib inline

print("--- Step 1 & 2: Loading and Merging Data ---")

# Load your dataset
df = pd.read_csv('../1_datasets/cleaned_datasets/clean_merged_data.csv')

# Load country coordinates dataset from a public source
country_coords_url = 'https://raw.githubusercontent.com/google/dspl/master/samples/google/canonical/countries.csv'
country_coords = pd.read_csv(country_coords_url)

# Keep only relevant columns and rename for merging
country_coords = country_coords[['name', 'latitude', 'longitude']]
country_coords = country_coords.rename(columns={'name': 'Country'})

# Merge on Country column
df_merged = pd.merge(df, country_coords, on='Country', how='left')

# Remove rows without coordinates
df_merged.dropna(subset=['latitude', 'longitude'], inplace=True)

# Display merged data
df_merged.head()


--- Step 1 & 2: Loading and Merging Data ---


FileNotFoundError: [Errno 2] No such file or directory: '1_datasets/cleaned_datasets/clean_merged_data.csv'

In [12]:
import os
os.getcwd()

'c:\\Users\\saade\\OneDrive\\Desktop\\CDSP\\ET6-CDSP-group-09-repo\\4_data_analysis'

In [5]:
import sys
print(sys.executable)

c:\Users\saade\anaconda3\python.exe


In [6]:
!c:\Users\saade\anaconda3\python.exe -m pip install folium


In [7]:
import folium

## Dataset Info

In [ ]:
# Show general info about the merged dataset
df_merged.info()

## Step 3: Standardize Features for Clustering

In [ ]:
print("\n--- Step 3: Preparing Data for K-means ---")

# Select relevant features
features = ['SDI', 'PM2.5', 'All-cause DALYs', 'Cardiovascular DALYs',
            'Stroke DALYs', 'Respiratory DALYs', 'latitude', 'longitude']

X = df_merged[features].copy()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame for readability
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

X_scaled_df.head()


##  Step 4: Determine Optimal Number of Clusters 

In [ ]:
print("\n--- Step 4: Determining Optimal K using Elbow Method ---")

sse = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    sse.append(kmeans.inertia_)

# Plot the SSE vs K graph
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), sse, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Sum of Squared Errors (SSE)")
plt.title("Elbow Method for Optimal K")
plt.xticks(range(1, 11))
plt.grid(True)
plt.show()


## Step 6: Visualize Clusters on World Map 

In [ ]:
print("\n--- Step 6: Visualizing Clusters on World Map ---")

# Create base map
m = folium.Map(location=[0, 0], zoom_start=2)

# Define color palette
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred',
          'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white',
          'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

# Add points by cluster
for idx, row in df_merged.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color=colors[row['Cluster'] % len(colors)],
        fill=True,
        fill_color=colors[row['Cluster'] % len(colors)],
        fill_opacity=0.7,
        tooltip=folium.Tooltip(
            f"Country: {row['Country']}<br>Year: {row['Year']}<br>Cluster: {row['Cluster']}<br>"
            f"SDI: {row['SDI']}<br>PM2.5: {row['PM2.5']}")
    ).add_to(m)

# Save and optionally display the map
map_file_path = 'world_map_clusters.html'
m.save(map_file_path)
m  # This will display the map in Jupyter (if supported)
